# Residential Composting Rate by Community District Over Time
**A basic spatiotemporal analysis of NYC DSNY Monthly Tonnage Data**

Source: [DSNY Monthly Tonnage Data](https://data.cityofnewyork.us/City-Government/DSNY-Monthly-Tonnage-Data/ebb7-mvp5/about_data) (dataset id `ebb7-mvp5`)

**What we compute.** There is no "composting rate" column in this data — we build it. For each community district and each month:

$$\text{composting rate} = \frac{\text{residential organics tons}}{\text{refuse} + \text{paper} + \text{MGP} + \text{residential organics}} \times 100$$

- **Spatial** dimension = `communitydistrict` (the finest grain in the data)
- **Temporal** dimension = `month`

> **What counts as "residential organics"?** We sum `resorganicstons` (curbside food + food-soiled paper), `leavesorganictons`, `xmastreetons`, and `otherorganicstons`, and **exclude** `schoolorganictons` (institutional, not residential). Because leaves and Christmas trees are bundled in, the rate is **seasonal** — expect fall (leaves) and December (trees) spikes. Set `INCLUDE_YARD_WASTE = False` below for a food-focused view.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## 1. Pull the data from the Socrata API

No download needed. **The one gotcha:** Socrata returns only 1,000 rows unless you raise `$limit`. This dataset has tens of thousands of district-month rows, so we set a high limit.

In [ ]:
DATASET_ID = "ebb7-mvp5"
url = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.csv?$limit=200000"

df = pd.read_csv(url)
print("Rows, columns:", df.shape)
df.head()

## 2. Define our column groups

Using the dataset's exact field names. The residential organics group is the heart of the analysis.

In [ ]:
TIME_COL  = "month"
BORO_COL  = "borough"
DIST_COL  = "communitydistrict"

REFUSE = "refusetonscollected"
PAPER  = "papertonscollected"
MGP    = "mgptonscollected"

# Residential organics streams (school organics deliberately excluded)
INCLUDE_YARD_WASTE = True
if INCLUDE_YARD_WASTE:
    ORG_COLS = ["resorganicstons", "leavesorganictons", "xmastreetons", "otherorganicstons"]
else:
    # food-focused: drop the clearly-seasonal yard streams
    ORG_COLS = ["resorganicstons", "otherorganicstons"]

RECYCLING = [REFUSE, PAPER, MGP]
ALL_STREAMS = RECYCLING + ORG_COLS
print("Numerator (organics):", ORG_COLS)
print("Denominator (all):   ", ALL_STREAMS)

## 3. Clean: coerce numbers and parse the month into a real date

Socrata sometimes returns tonnage as text, so we force it numeric. We also turn `month` (e.g. `"2024 / 07"`) into an actual `datetime` so sorting and time-series ops work.

In [ ]:
# Coerce tonnage columns to numeric
for c in ALL_STREAMS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Parse the month field into a first-of-month timestamp.
# Handles "YYYY / MM", "YYYY-MM", etc. If your format differs, tweak the regex.
def parse_month(s):
    m = re.search(r"(\d{4})\D+(\d{1,2})", str(s))
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), 1) if m else pd.NaT

df["date"] = df[TIME_COL].map(parse_month)
df = df.dropna(subset=["date"]).sort_values("date")

print("Date range:", df["date"].min().date(), "->", df["date"].max().date())
print("Districts: ", df[DIST_COL].nunique())
df[[TIME_COL, "date"]].drop_duplicates().head()

## 4. Build the streams and the composting rate

Sum organics and total per row, then take organics as a share of everything collected.

In [ ]:
df["organics"] = df[ORG_COLS].sum(axis=1)
df["total"]    = df[ALL_STREAMS].sum(axis=1)

df["compost_rate"] = np.where(df["total"] > 0,
                              df["organics"] / df["total"] * 100,
                              np.nan)

df[[DIST_COL, "date", "organics", "total", "compost_rate"]].head()

## 5. Temporal view — citywide composting rate over time

We aggregate correctly: **sum tons across districts first, then divide.** (Averaging the per-district percentages would weight a tiny district the same as a huge one.)

In [ ]:
city = (df.groupby("date")
          .apply(lambda g: g["organics"].sum() / g["total"].sum() * 100,
                 include_groups=False)
          .rename("compost_rate"))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(city.index, city.values, marker="o", ms=4, color="#2e7d32")
ax.set_title("NYC citywide residential composting rate over time")
ax.set_ylabel("Composting rate (%)")
ax.set_xlabel("Month")
ax.grid(alpha=0.3)
plt.show()

city.tail(12).round(2)

## 6. Spatial + temporal — one line per borough

The outer boroughs (yards, single-family homes) typically sit higher; Manhattan tends to lag.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for boro, g in df.groupby(BORO_COL):
    s = (g.groupby("date")
           .apply(lambda x: x["organics"].sum() / x["total"].sum() * 100,
                  include_groups=False))
    ax.plot(s.index, s.values, marker="o", ms=3, label=boro)

ax.set_title("Residential composting rate by borough over time")
ax.set_ylabel("Composting rate (%)")
ax.set_xlabel("Month")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 7. The full spatiotemporal matrix — district × month heatmap

Every community district as a row, every month as a column. Bright = higher composting. You can read both the rollout over time (left-to-right brightening) and the neighborhood gradient (top-to-bottom).

In [ ]:
pivot = df.pivot_table(index=DIST_COL, columns="date", values="compost_rate")

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlGn")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=7)
step = max(1, len(pivot.columns) // 24)
ax.set_xticks(range(0, len(pivot.columns), step))
ax.set_xticklabels([d.strftime("%Y-%m") for d in pivot.columns[::step]],
                   rotation=45, ha="right", fontsize=8)
fig.colorbar(im, label="Composting rate (%)")
ax.set_title("Composting rate: community district x month")
plt.tight_layout()
plt.show()

## 8. Which districts changed most? (year-over-year)

Rank districts by the change between two **same-month** dates a year apart — comparing like months controls for seasonality. Edit the two dates to fit your data range.

In [ ]:
def rate_by_district(date_str):
    sub = df[df["date"] == pd.Timestamp(date_str)]
    return (sub.groupby(DIST_COL)
               .apply(lambda g: g["organics"].sum() / g["total"].sum() * 100,
                      include_groups=False))

# EDIT these to two same-month dates that exist in your data:
EARLIER, LATER = "2024-06-01", "2025-06-01"

try:
    change = (rate_by_district(LATER) - rate_by_district(EARLIER)).sort_values(ascending=False).round(2)
    print(f"Composting-rate change (percentage points), {EARLIER} -> {LATER}\n")
    print("Biggest increases:\n", change.head(10).to_string())
    print("\nBiggest decreases:\n", change.tail(10).to_string())
except Exception:
    print("Adjust EARLIER/LATER to dates in your range:",
          df["date"].min().date(), "->", df["date"].max().date())

## Caveats & next steps

**Caveats**
- **"Organics" ≠ pure food composting.** It bundles yard waste and Christmas trees, hence seasonal spikes. Toggle `INCLUDE_YARD_WASTE = False` in Step 2 for a food-focused view.
- **This is a diversion-style rate, not a true capture rate.** It's organics as a share of what was *collected*. The true capture rate (organics ÷ compostable material *generated*, including compostables still in the refuse bin) needs the **2023 Waste Characterization Study** — residential waste is ~36% compostable — multiplied against the refuse tonnage to form the denominator.
- **Geography** is the sanitation community district — operationally defined, won't join perfectly to census tracts/ZIPs.
- **Residential focus:** we excluded `schoolorganictons`; the municipal stream is predominantly residential.

**Next steps**
1. Join a community-district shapefile (GeoPandas) for a true choropleth map instead of a heatmap.
2. Bring in the ~36% compostable fraction to compute an actual capture rate.
3. Overlay the enforcement timeline (mandate Oct 2024; citywide fines Jan 2026) to see policy effects.